# MLI Synthetics - Kaggle Runner (llama-cpp / Mistral-Nemo Q4_K_M)

Phase 1 pipeline on Kaggle T4. The repo's HFClient is overwritten in cell 2
with a llama-cpp-python implementation that runs the Q4_K_M GGUF on GPU.

**HFClient uses a class-level singleton** (`_instance` lives on the
class object) so the model loads into VRAM exactly once even if analyzer
+ designer both call the factory.


## Zelle 1: Setup


In [ ]:
# Repo klonen und Dependencies installieren
!git clone https://github.com/thekuhldude/mli-synthetics
%cd /kaggle/working/mli-synthetics

!pip install -e ".[dev]" -q
!pip install librosa soundfile pypdf llama-cpp-python -q

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


## Zelle 2: HFClient ueberschreiben (llama-cpp, class-level Singleton)


In [ ]:
%%writefile src/mli_synthetics/llm/hf_client.py
"""llama-cpp-python based HFClient for Mistral-Nemo GGUF (class singleton).

Drop-in replacement for the transformers-based hf_client.py. Used on
Kaggle where llama-cpp-python is the fastest path to running
Mistral-Nemo-Instruct-2407 Q4_K_M on a T4. Same async `generate`
interface as OllamaClient so the rest of the pipeline doesn't change.

Singleton via class attributes: `_instance`/`_model`/`_tokenizer`/
`_pipeline` live on the class object, not on a module-level variable,
so they survive across import patterns and the 12B GGUF is loaded
into VRAM exactly once per process.
"""
from __future__ import annotations

import asyncio
import json
import re
from pathlib import Path
from typing import Any

from mli_synthetics.errors import (
    OllamaConnectionError,
    OllamaError,
    OllamaInvalidJSONError,
    OllamaModelNotFoundError,
)
from mli_synthetics.logging_config import get_logger

logger = get_logger()

DEFAULT_REPO = "bartowski/Mistral-Nemo-Instruct-2407-GGUF"
DEFAULT_FILE = "Mistral-Nemo-Instruct-2407-Q4_K_M.gguf"


class HFClient:
    """llama-cpp-python wrapper exposing the OllamaClient interface.

    Class-level singleton: every `HFClient()` call returns the same
    instance and the model loads exactly once.
    """

    _instance: "HFClient | None" = None
    _model: Any = None
    _tokenizer: Any = None
    _pipeline: Any = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, base_url: str = "", timeout: int = 900, **kwargs: Any):
        # Already initialized - nothing to do.
        if getattr(self, "_ready", False):
            return
        self._ready = True
        self.repo_id = DEFAULT_REPO
        self.filename = DEFAULT_FILE
        self.base_url = base_url
        self.timeout = timeout
        self.model_id = f"{DEFAULT_REPO}/{DEFAULT_FILE}"

    # ------------------------------------------------------------------
    def _load(self) -> None:
        if type(self)._model is not None:
            return
        try:
            from llama_cpp import Llama
        except ImportError as exc:
            raise OllamaConnectionError(
                "llama-cpp-python required. Install with: "
                "pip install llama-cpp-python"
            ) from exc
        logger.info("HFClient: loading {} ({})", self.repo_id, self.filename)
        try:
            type(self)._model = Llama.from_pretrained(
                repo_id=self.repo_id,
                filename=self.filename,
                n_gpu_layers=-1,
                n_ctx=16384,
                verbose=False,
            )
        except Exception as exc:  # noqa: BLE001
            msg = str(exc)
            if "404" in msg or "not found" in msg.lower():
                raise OllamaModelNotFoundError(
                    f"GGUF file '{self.filename}' not found in {self.repo_id}"
                ) from exc
            raise OllamaConnectionError(f"GGUF model load failed: {exc}") from exc

    # ------------------------------------------------------------------
    async def health_check(self) -> bool:
        try:
            import llama_cpp  # noqa: F401
            return True
        except ImportError:
            return False

    async def list_models(self) -> list[str]:
        return [self.model_id]

    # ------------------------------------------------------------------
    async def generate(
        self,
        model: str | None = None,
        prompt: str = "",
        messages: list[dict] | None = None,
        system: str | None = None,
        temperature: float = 0.7,
        max_tokens: int = 1500,
        json_mode: bool = False,
        **kwargs: Any,
    ) -> str:
        del model

        sys_text = system if system is not None else kwargs.get("system", "")
        sys_text = (sys_text or "").rstrip()
        if json_mode:
            sys_text = (
                sys_text + "\n\nRespond with ONLY valid JSON, no markdown, no commentary."
            ).strip()

        user_text = prompt
        if not user_text and messages:
            for m in messages:
                if m.get("role") == "user":
                    user_text = m.get("content", "")
                    break

        formatted = f"[INST] <<SYS>>\n{sys_text}\n<</SYS>>\n\n{user_text} [/INST]"

        loop = asyncio.get_event_loop()
        try:
            raw = await asyncio.wait_for(
                loop.run_in_executor(
                    None, lambda: self._run(formatted, temperature, max_tokens)
                ),
                timeout=float(self.timeout),
            )
        except asyncio.TimeoutError as exc:
            raise OllamaConnectionError(
                "LLM took too long. Try a smaller model or split the input."
            ) from exc

        cleaned = raw.strip()
        if json_mode:
            cleaned = _extract_json(cleaned)
            try:
                json.loads(cleaned)
            except json.JSONDecodeError:
                logger.warning("HFClient: JSON parse failed, retrying once at lower temp")
                try:
                    raw2 = await asyncio.wait_for(
                        loop.run_in_executor(
                            None,
                            lambda: self._run(
                                formatted, max(0.1, temperature - 0.3), max_tokens
                            ),
                        ),
                        timeout=float(self.timeout),
                    )
                except asyncio.TimeoutError as exc2:
                    raise OllamaConnectionError(
                        "LLM took too long. Try a smaller model or split the input."
                    ) from exc2
                cleaned = _extract_json(raw2.strip())
                try:
                    json.loads(cleaned)
                except json.JSONDecodeError as exc3:
                    raise OllamaInvalidJSONError(
                        f"GGUF model did not return valid JSON after retry: {exc3}"
                    ) from exc3
        return cleaned

    # ------------------------------------------------------------------
    def _run(self, formatted_prompt: str, temperature: float, max_tokens: int) -> str:
        self._load()
        model = type(self)._model
        assert model is not None
        out = model(
            formatted_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            stop=["[/INST]", "</s>"],
            echo=False,
        )
        if isinstance(out, dict):
            choices = out.get("choices", [])
            if choices:
                return str(choices[0].get("text", ""))
        return str(out)


# ---------------------------------------------------------------------------
def _extract_json(text: str) -> str:
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    first = text.find("{")
    last = text.rfind("}")
    if first != -1 and last != -1 and last > first:
        return text[first : last + 1]
    return text


__all__ = [
    "HFClient",
    "DEFAULT_REPO",
    "DEFAULT_FILE",
    "OllamaConnectionError",
    "OllamaError",
    "OllamaInvalidJSONError",
    "OllamaModelNotFoundError",
]


## Zelle 3: Path & Environment Setup


In [ ]:
import sys
import os

sys.path.insert(0, '/kaggle/working/mli-synthetics/src')
os.environ["USE_HF_CLIENT"] = "true"

import nest_asyncio
nest_asyncio.apply()


## Zelle 4: Modell-Test


In [ ]:
import asyncio
from mli_synthetics.llm.hf_client import HFClient

client = HFClient()

async def test():
    result = await client.generate(prompt="Say hello in one word")
    print("Model test:", result)

asyncio.get_event_loop().run_until_complete(test())


## Zelle 5: Audio-Datei finden


In [ ]:
import subprocess
result = subprocess.run(
    ["find", "/kaggle/input", "-name", "*.wav"],
    capture_output=True, text=True
)
wav_files = [f for f in result.stdout.strip().split("\n") if f]
print("Gefundene WAV-Dateien:")
for f in wav_files:
    print(f"  {f}")

if not wav_files:
    raise FileNotFoundError(
        "Keine WAV-Datei unter /kaggle/input gefunden. "
        "Mounte ein Audio-Dataset bevor du diese Zelle laufen laesst."
    )

# Erste Datei als Default nehmen
AUDIO_PATH = wav_files[0]
print(f"\nVerwende: {AUDIO_PATH}")


## Zelle 6: Pipeline ausfuehren


In [ ]:
from pathlib import Path
from mli_synthetics.pipeline.orchestrator import Phase1Pipeline

audio_path = Path(AUDIO_PATH)
output_dir = Path("/kaggle/working/outputs")
output_dir.mkdir(exist_ok=True)

pipeline = Phase1Pipeline()

async def run_pipeline():
    result = await pipeline.generate_show(
        audio_path=audio_path,
        output_dir=output_dir,
    )
    print("\nPipeline complete!")
    print(result)
    return result

result = asyncio.get_event_loop().run_until_complete(run_pipeline())


## Zelle 7: Output zippen


In [ ]:
import shutil
zip_path = shutil.make_archive(
    '/kaggle/working/mli_output',
    'zip',
    '/kaggle/working/outputs'
)
print(f"Download: {zip_path}")
print("Rechts in Sidebar: 'Output' Tab -> mli_output.zip -> Download")
